[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/1-getting-started/03_saving_and_loading_indexes.ipynb)

# Saving & Loading Indexes

So far, every notebook has rebuilt the index from scratch by re-running `TableIndexer.create_index()`. That's fine for a notebook, but wasteful for a real service, you don't want to recompile from a DataFrame every time your application restarts.

M|BOX lets you compile once, save the result to a single binary archive, and load it back in milliseconds, no source DataFrame required at load time.

In this notebook you will:

1. Rebuild the index from the previous notebook
2. Save it to disk with `to_binary()`
3. Load it back with `load_binary()`, in a way that simulates a fresh process
4. Confirm the loaded index behaves identically to the original

## 1. Rebuild the index

Same dataset and same setup as the previous notebook, typos and all. If you've already got `index` in memory from `02_loading_your_own_csv.ipynb`, this just re-creates it so this notebook can run standalone.

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer

csv_contents = """customer_id,full_name,city,address,age,source_system
C2001,Jonathan Reevs,Chiago,742 Evergreen Terace,37,CRM_LEGACY
C2002,Priyanka Chopra,Austin,150 Riverside Drive,29,CRM_LEGACY
C2003,Muhamad Al-Farsi,Dubai,12 Sheik Zayed Rd,45,CRM_NEW
C2004,Elanor Whitfield,Dublin,9 St. Stephens Grn,52,CRM_NEW
"""

with open("datasets/customers.csv", "w", encoding="utf-8") as f:
    f.write(csv_contents)

df = pd.read_csv("datasets/customers.csv")

index = TableIndexer.create_index(
    df=df,
    index_columns=["full_name", "city", "address"],
    tmp_dir="tmp_index"
)

index

## 2. Save the index to disk

`TableIndex.to_binary()` compiles the index state, including its schema, harmonization rules, and payload data, into a single `.zip` archive.

In [3]:
saved = index.to_binary("./customer_index.zip")
print("Saved successfully:", saved)

Saved successfully: True


> 💡 The `.zip` archive contains everything needed to reconstruct the index, you don't need to keep the original CSV or DataFrame around once this file exists. That makes it safe to check into a build artifact store or deploy alongside a service, without also shipping the raw source data.

## 3. Load the index in a "fresh" process

To make the point clearly: let's delete the in-memory `index` and `df` entirely, so nothing but the `.zip` file on disk remains. This simulates what happens when a new process, a restarted API server, a fresh container, a different machine, starts up and needs to be ready to match immediately.

In [4]:
del index
del df

# At this point, nothing is in memory. Only customer_index.zip exists on disk.
import os
print("File on disk:", os.path.exists("./customer_index.zip"))

File on disk: True


In [5]:
from mbox.indexing import TableIndex

loaded_index = TableIndex.load_binary("./customer_index.zip")
loaded_index

No DataFrame, no `TableIndexer`, no rebuilding, `loaded_index` is immediately ready to query, exactly as if it had just been compiled.

## 4. Confirm it behaves identically

Let's run the same query from the previous notebook against the freshly loaded index.

In [6]:
results = loaded_index.match(
    full_name="Jonathan Reeves",
    city="Chicago",
    address="742 Evergreen Terrace",
    include_field_scores=True
)

results

,query_row,index_row,full_name_candidate,city_candidate,address_candidate,customer_id_candidate,age_candidate,source_system_candidate,overall_score,full_name_score,city_score,address_score
0,0,0,Jonathan Reevs,Chiago,742 Evergreen Terace,C2001,37,CRM_LEGACY,80,83,69,88


Same candidate, same scores, same explainability, the loaded index is functionally identical to the one we compiled and saved. You can also confirm the schema itself came along for the ride:

In [7]:
loaded_index.describe()

,Field,Index Type,Unique Value Count,Index size [bytes]
0,full_name,IndexType.PHRASE,4,26776
1,city,IndexType.TERM,4,9516
2,address,IndexType.PHRASE,4,27704
3,customer_id,IndexType.NON_SEARCHABLE,4,0
4,age,IndexType.NON_SEARCHABLE,4,0
5,source_system,IndexType.NON_SEARCHABLE,4,0


`full_name` and `address` are still `IndexType.PHRASE`, `city` is still `IndexType.TERM`, and the `NON_SEARCHABLE` payload fields are all intact, none of that had to be re-specified after loading.

## Next steps

- **`2-data-harmonization/`**, attach `CharacterMapping` and `AliasSet` rules so accents, umlauts, and nicknames resolve automatically, before you compile and save
- **`6-production/benchmarking_index_size_and_latency.ipynb`**, see how index size and load time scale with real dataset sizes
- **`7-integrations/mbox_in_a_fastapi_service.ipynb`**, a realistic example of loading a saved index at service startup

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*